# Parameter Sweep Analysis

**Objective**: Systematically explore how key parameters affect simulation outcomes, with focus on:
- Destination concentration (Gini coefficient)
- Segment heterogeneity (coefficient of variation)
- TFI dynamics (policy stress)
- Segment-specific behaviors

**Method**: One-at-a-time (OAT) sensitivity analysis - freeze all parameters at baseline, sweep one parameter across its theoretical range.

---

## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from datetime import datetime
import importlib

# Plotting configuration
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print(f"Analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

### Baseline Configuration

Parameters frozen at literature-backed/calibrated values:

In [ ]:
# Baseline parameters (frozen for OAT sweeps)
BASELINE_CONFIG = {
    # Simulation setup
    'agent_count': 4000,
    'duration_days': 365,
    'seed': 42,
    'segment_shares': {
        'budget': 0.30,
        'luxury': 0.20,
        'adventure': 0.25,
        'family': 0.25,
    },
    
    # Utility weights (from utility.py)
    'softmax_temperature': 1.0,  # Will be swept
    'distance_friction': 1.0,    # Multiplier on distance weights
    
    # Destination dynamics (from destination.py)
    'capacity_threshold': 0.80,  # Will be swept
    'tfi_decline_rate': 0.05,    # Will be swept
    'tfi_recovery_rate': 0.02,
    
    # Crowding feedback
    'crowding_exponent': 2.0,    # Will be swept
}

print("Baseline Configuration:")
for key, value in BASELINE_CONFIG.items():
    if key != 'segment_shares':
        print(f"  {key}: {value}")

### Parameter Sweep Ranges

Each parameter will be swept across this range while others remain at baseline:

In [ ]:
# Parameter sweep definitions
PARAM_SWEEPS = {
    'softmax_temperature': {
        'range': [0.3, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5],
        'baseline': 1.0,
        'label': 'Softmax Temperature (τ)',
        'description': 'Controls exploration vs exploitation in destination choice',
    },
    'capacity_threshold': {
        'range': [0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95],
        'baseline': 0.80,
        'label': 'Capacity Threshold',
        'description': 'Crowding ratio that triggers TFI decline',
    },
    'tfi_decline_rate': {
        'range': [0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10],
        'baseline': 0.05,
        'label': 'TFI Decline Rate',
        'description': 'Rate of TFI decline when crowding exceeds threshold',
    },
    'distance_friction': {
        'range': [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0],
        'baseline': 1.0,
        'label': 'Distance Friction Multiplier',
        'description': 'Scaling factor on distance penalty in utility function',
    },
    'crowding_exponent': {
        'range': [1.0, 1.5, 1.75, 2.0, 2.25, 2.5, 3.0],
        'baseline': 2.0,
        'label': 'Crowding Exponent (θ)',
        'description': 'Non-linearity of crowding disutility',
    },
}

print("Parameter Sweep Ranges:")
for param_name, config in PARAM_SWEEPS.items():
    print(f"\n  {config['label']}:")
    print(f"    Range: {config['range']}")
    print(f"    Baseline: {config['baseline']}")

## 2. Metrics Calculation

In [ ]:
def calculate_gini(values: List[float]) -> float:
    """Calculate Gini coefficient for destination concentration."""
    if len(values) < 2 or sum(values) == 0:
        return 0.0
    
    sorted_values = sorted(values)
    n = len(sorted_values)
    cumulative = np.cumsum(sorted_values)
    gini = (2 * np.sum((np.arange(1, n + 1) * sorted_values))) / (n * np.sum(sorted_values)) - (n + 1) / n
    return gini


def calculate_heterogeneity_cv(segment_metrics: Dict[str, float]) -> float:
    """
    Calculate coefficient of variation across segments.
    
    Measures how differently segments behave (higher = more differentiation).
    """
    values = list(segment_metrics.values())
    if len(values) < 2:
        return 0.0
    
    mean = np.mean(values)
    if mean == 0:
        return 0.0
    
    std = np.std(values)
    return std / mean


def calculate_metrics(sim, param_name: str, param_value: float, seed: int) -> Dict:
    """
    Calculate comprehensive metrics from simulation run.
    
    Returns both global and segment-specific metrics.
    """
    dc = sim.data_collector
    
    # 1. Gini coefficient (final 30-day average)
    gini_values = []
    for tick in range(max(0, len(dc.global_active) - 30), len(dc.global_active)):
        values = []
        for code, history in dc.dest_visitors.items():
            if tick < len(history):
                values.append(history[tick])
        if len(values) > 1 and sum(values) > 0:
            gini_values.append(calculate_gini(values))
    gini_avg = np.mean(gini_values) if gini_values else 0.0
    
    # 2. Total arrivals
    total_arrivals = sum(dc.global_active)
    
    # 3. TFI decline (% destinations with TFI < 0.60)
    tfi_stressed = sum(1 for d in sim.destinations.values() if d.tfi < 0.60)
    tfi_stress_pct = (tfi_stressed / len(sim.destinations)) * 100 if sim.destinations else 0.0
    
    # 4. Top 10 destination share
    final_visitors = {}
    for code, history in dc.dest_visitors.items():
        if history:
            final_visitors[code] = history[-1]
    total_final = sum(final_visitors.values())
    top10_share = 0.0
    if total_final > 0:
        top10 = sorted(final_visitors.values(), reverse=True)[:10]
        top10_share = (sum(top10) / total_final) * 100
    
    # 5. Segment-specific metrics
    segment_distances = {}
    segment_concentration = {}
    
    for segment in ['budget', 'luxury', 'adventure', 'family']:
        # Average distance traveled (from collected data)
        if hasattr(dc, 'segment_trips') and segment in dc.segment_trips:
            trips = dc.segment_trips[segment]
            if trips:
                avg_distance = np.mean([t.get('distance', 0) for t in trips])
                segment_distances[segment] = avg_distance
            else:
                segment_distances[segment] = 0.0
        else:
            segment_distances[segment] = 0.0
        
        # Destination concentration per segment
        if hasattr(dc, 'segment_dest_counts') and segment in dc.segment_dest_counts:
            dest_counts = dc.segment_dest_counts[segment]
            if dest_counts:
                concentration = calculate_gini(list(dest_counts.values()))
                segment_concentration[segment] = concentration
            else:
                segment_concentration[segment] = 0.0
        else:
            segment_concentration[segment] = 0.0
    
    # 6. Heterogeneity (CV across segments)
    heterogeneity_distance = calculate_heterogeneity_cv(segment_distances)
    heterogeneity_concentration = calculate_heterogeneity_cv(segment_concentration)
    
    # 7. Average trip length by segment
    segment_trip_lengths = {}
    for segment in ['budget', 'luxury', 'adventure', 'family']:
        if hasattr(dc, 'segment_trip_lengths') and segment in dc.segment_trip_lengths:
            lengths = dc.segment_trip_lengths[segment]
            segment_trip_lengths[segment] = np.mean(lengths) if lengths else 0.0
        else:
            segment_trip_lengths[segment] = 0.0
    
    return {
        'param_name': param_name,
        'param_value': param_value,
        'seed': seed,
        'gini_coefficient': gini_avg,
        'total_arrivals': total_arrivals,
        'tfi_stress_pct': tfi_stress_pct,
        'top10_share': top10_share,
        'segment_distances': segment_distances,
        'segment_concentration': segment_concentration,
        'heterogeneity_distance': heterogeneity_distance,
        'heterogeneity_concentration': heterogeneity_concentration,
        'segment_trip_lengths': segment_trip_lengths,
    }

## 3. Simulation Runner

In [ ]:
def apply_parameter_patch(param_name: str, param_value: float):
    """
    Apply parameter modification to simulation modules.
    
    Patches module-level constants before importing simulation.
    """
    # Force fresh imports
    modules_to_remove = [k for k in sys.modules.keys() if k.startswith('simulation')]
    for mod in modules_to_remove:
        del sys.modules[mod]
    
    if param_name == 'tfi_decline_rate':
        import simulation.destinations.destination as dest_module
        dest_module.TFI_DECLINE_RATE = param_value
        
    elif param_name == 'capacity_threshold':
        import simulation.destinations.destination as dest_module
        dest_module.CROWDING_THRESHOLD = param_value
        
    elif param_name == 'softmax_temperature':
        import simulation.mechanics.choice as choice_module
        choice_module.SOFTMAX_TEMPERATURE = param_value
        # Also patch segment-specific temperatures
        import simulation.mechanics.utility as utility_module
        for segment in utility_module.SEGMENT_TEMPERATURE:
            utility_module.SEGMENT_TEMPERATURE[segment] = param_value
        
    elif param_name == 'distance_friction':
        import simulation.mechanics.distance as dist_module
        if hasattr(dist_module, 'DISTANCE_WEIGHTS'):
            # Store original weights if not already patched
            if not hasattr(dist_module, '_original_weights'):
                dist_module._original_weights = dist_module.DISTANCE_WEIGHTS.copy()
            # Apply multiplier to original weights
            for segment in dist_module._original_weights:
                dist_module.DISTANCE_WEIGHTS[segment] = dist_module._original_weights[segment] * param_value
        
    elif param_name == 'crowding_exponent':
        # Patch the crowding exponent in utility calculation
        import simulation.mechanics.utility as utility_module
        utility_module.CROWDING_EXPONENT = param_value


def run_simulation_with_params(
    param_name: str,
    param_value: float,
    countries_data: List[Dict],
    config: Dict = None,
) -> Dict:
    """
    Run single simulation with modified parameter.
    
    Returns metrics dictionary.
    """
    # Apply parameter patch
    apply_parameter_patch(param_name, param_value)
    
    # Import and run simulation
    from simulation.simulation import Simulation
    from simulation.data.loaders import load_country_data
    
    sim_config = {
        'duration_days': BASELINE_CONFIG['duration_days'],
        'agent_count': BASELINE_CONFIG['agent_count'],
        'seed': BASELINE_CONFIG['seed'],
        'segment_shares': BASELINE_CONFIG['segment_shares'],
        **(config or {})
    }
    
    sim = Simulation(
        countries_data=countries_data,
        config=sim_config,
    )
    sim.initialize()
    sim.run(BASELINE_CONFIG['duration_days'])
    
    return calculate_metrics(sim, param_name, param_value, BASELINE_CONFIG['seed'])

## 4. Execute Parameter Sweeps

In [ ]:
# Skip this cell if you loaded results from CSV above
if 'df_results' in locals() and len(df_results) > 0:
    print("Results already loaded from CSV, skipping execution...")
    print("To re-run the sweep, restart the kernel and run from the beginning.")
else:
    # Execute the sweep
    print("No results loaded, executing parameter sweeps...")
print("=" * 80)
print("PARAMETER SWEEP ANALYSIS")
print("=" * 80)
print(f"\nStart time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nBaseline configuration:")
print(f"  Agents: {BASELINE_CONFIG['agent_count']}")
print(f"  Duration: {BASELINE_CONFIG['duration_days']} days")
print(f"  Seed: {BASELINE_CONFIG['seed']}")
print(f"\nParameters to sweep: {len(PARAM_SWEEPS)}")
for param_name, config in PARAM_SWEEPS.items():
    print(f"  - {config['label']}: {len(config['range'])} values")

# Load country data (once)
print("\n[1/2] Loading country data...")
from simulation.data.loaders import load_country_data
countries = load_country_data()
print(f"  Loaded {len(countries)} countries")

# Run parameter sweeps
print("\n[2/2] Running parameter sweeps...")
all_results = []

total_runs = sum(len(config['range']) for config in PARAM_SWEEPS.values())
run_count = 0

for param_name, sweep_config in PARAM_SWEEPS.items():
    print(f"\n  Sweeping: {sweep_config['label']}")
    
    for param_value in sweep_config['range']:
        run_count += 1
        print(f"    [{run_count}/{total_runs}] {param_name} = {param_value}...", end=" ", flush=True)
        
        try:
            metrics = run_simulation_with_params(
                param_name=param_name,
                param_value=param_value,
                countries_data=countries,
            )
            all_results.append(metrics)
            print(f"Gini={metrics['gini_coefficient']:.3f}, Heterogeneity={metrics['heterogeneity_distance']:.3f}")
            
        except Exception as e:
            print(f"ERROR: {e}")
            import traceback
            traceback.print_exc()

print(f"\n\nCompleted {len(all_results)} runs")
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 5. Results Analysis

In [ ]:
# Load results from CSV (preferred) or use all_results variable
import os
csv_path = Path('../docs/validation/parameter_sweep_results.csv')

if csv_path.exists():
    print(f"Loading results from {csv_path}...")
    df_results = pd.read_csv(csv_path)
    print(f"Loaded {len(df_results)} runs from CSV")
else:
    # Fallback to all_results if CSV doesn't exist
    print("CSV not found, converting from all_results...")
    results_flat = []
    for r in all_results:
        row = {
            'param_name': r['param_name'],
            'param_value': r['param_value'],
            'gini': r['gini_coefficient'],
            'total_arrivals': r['total_arrivals'],
            'tfi_stress_pct': r['tfi_stress_pct'],
            'top10_share': r['top10_share'],
            'heterogeneity_distance': r['heterogeneity_distance'],
            'heterogeneity_concentration': r['heterogeneity_concentration'],
        }
        # Add segment-specific metrics
        for segment in ['budget', 'luxury', 'adventure', 'family']:
            row[f'{segment}_distance'] = r['segment_distances'].get(segment, 0)
            row[f'{segment}_concentration'] = r['segment_concentration'].get(segment, 0)
            row[f'{segment}_trip_length'] = r['segment_trip_lengths'].get(segment, 0)
        results_flat.append(row)
    
    df_results = pd.DataFrame(results_flat)
    # Save to CSV for future use
    df_results.to_csv(csv_path, index=False)
    print(f"Saved results to {csv_path}")

print(f"\nDataFrame shape: {df_results.shape}")
print(f"Columns: {list(df_results.columns)}")
df_results.head()


### 5.1 Sensitivity Ranking

Which parameters have the strongest effect on key metrics?

In [ ]:
# Calculate sensitivity indices (range / baseline)
sensitivity_metrics = ['gini', 'heterogeneity_distance', 'tfi_stress_pct', 'top10_share']

sensitivity_data = []
for param_name in PARAM_SWEEPS.keys():
    param_data = df_results[df_results['param_name'] == param_name]
    
    for metric in sensitivity_metrics:
        values = param_data[metric].values
        baseline_val = param_data[param_data['param_value'] == PARAM_SWEEPS[param_name]['baseline']][metric].values
        
        if len(baseline_val) > 0 and baseline_val[0] > 0:
            range_val = max(values) - min(values)
            sensitivity_idx = range_val / baseline_val[0]
            
            sensitivity_data.append({
                'parameter': param_name,
                'metric': metric,
                'sensitivity_index': sensitivity_idx,
                'min': min(values),
                'max': max(values),
                'baseline': baseline_val[0],
            })

df_sensitivity = pd.DataFrame(sensitivity_data)

# Plot sensitivity ranking
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, metric in enumerate(sensitivity_metrics):
    metric_data = df_sensitivity[df_sensitivity['metric'] == metric].sort_values('sensitivity_index', ascending=True)
    
    ax = axes[i]
    bars = ax.barh(metric_data['parameter'], metric_data['sensitivity_index'], color='steelblue')
    ax.set_xlabel('Sensitivity Index (Range / Baseline)')
    ax.set_title(f'Sensitivity Ranking: {metric.replace("_", " ").title()}')
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Moderate sensitivity')
    ax.axvline(x=1.0, color='darkred', linestyle='--', alpha=0.5, label='High sensitivity')
    ax.legend()
    
    # Add value labels
    for bar, val in zip(bars, metric_data['sensitivity_index']):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('docs/validation/parameter_sweep_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSensitivity Ranking Summary:")
print(df_sensitivity.pivot(index='parameter', columns='metric', values='sensitivity_index').round(3))

### 5.2 Global Metrics vs Parameters

In [ ]:
# Plot global metrics vs each parameter
fig, axes = plt.subplots(len(PARAM_SWEEPS), 3, figsize=(15, 4 * len(PARAM_SWEEPS)))

metrics_to_plot = [
    ('gini', 'Gini Coefficient', 'Destination Concentration'),
    ('heterogeneity_distance', 'Heterogeneity (CV)', 'Segment Differentiation'),
    ('tfi_stress_pct', 'TFI Stress (%)', 'Policy Stress'),
]

for i, (param_name, sweep_config) in enumerate(PARAM_SWEEPS.items()):
    param_data = df_results[df_results['param_name'] == param_name].sort_values('param_value')
    baseline_val = sweep_config['baseline']
    
    for j, (metric_key, metric_label, title) in enumerate(metrics_to_plot):
        ax = axes[i, j] if len(PARAM_SWEEPS) > 1 else axes[j]
        
        ax.plot(param_data['param_value'], param_data[metric_key], 'o-', linewidth=2, markersize=8, color='navy')
        ax.axvline(x=baseline_val, color='red', linestyle='--', alpha=0.7, label=f'Baseline ({baseline_val})')
        ax.set_xlabel(sweep_config['label'])
        ax.set_ylabel(metric_label)
        ax.set_title(f'{title}\nvs {sweep_config["label"]}')
        ax.grid(True, alpha=0.3)
        
        # Mark optimal region (closest to validation target)
        if metric_key == 'gini':
            target = 0.71  # Validation target
            ax.axhline(y=target, color='green', linestyle=':', alpha=0.7, label=f'Target ({target})')
            ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('docs/validation/parameter_sweep_global_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Segment-Specific Analysis

In [ ]:
# Plot segment-specific metrics vs parameters
segments = ['budget', 'luxury', 'adventure', 'family']
segment_colors = {'budget': '#2E86AB', 'luxury': '#A23B72', 'adventure': '#F18F01', 'family': '#4CAF50'}

fig, axes = plt.subplots(len(PARAM_SWEEPS), 2, figsize=(14, 4 * len(PARAM_SWEEPS)))

for i, (param_name, sweep_config) in enumerate(PARAM_SWEEPS.items()):
    param_data = df_results[df_results['param_name'] == param_name].sort_values('param_value')
    baseline_val = sweep_config['baseline']
    
    # Left: Average distance by segment
    ax1 = axes[i, 0] if len(PARAM_SWEEPS) > 1 else axes[0]
    
    for segment in segments:
        col = f'{segment}_distance'
        ax1.plot(param_data['param_value'], param_data[col], 'o-', 
                linewidth=2, markersize=6, color=segment_colors[segment], 
                label=segment.capitalize())
    
    ax1.axvline(x=baseline_val, color='gray', linestyle='--', alpha=0.5, label='Baseline')
    ax1.set_xlabel(sweep_config['label'])
    ax1.set_ylabel('Average Distance (km)')
    ax1.set_title(f'Segment Distance\nvs {sweep_config["label"]}')
    ax1.legend(ncol=2, fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Right: Heterogeneity (CV) across segments
    ax2 = axes[i, 1] if len(PARAM_SWEEPS) > 1 else axes[1]
    
    ax2.plot(param_data['param_value'], param_data['heterogeneity_distance'], 
            's-', linewidth=2.5, markersize=10, color='darkblue')
    ax2.axvline(x=baseline_val, color='red', linestyle='--', alpha=0.7, label='Baseline')
    ax2.set_xlabel(sweep_config['label'])
    ax2.set_ylabel('Heterogeneity (CV)')
    ax2.set_title(f'Segment Heterogeneity\nvs {sweep_config["label"]}')
    ax2.grid(True, alpha=0.3)
    
    # Annotation
    max_het_idx = param_data['heterogeneity_distance'].idxmax()
    max_het_row = param_data.loc[max_het_idx]
    ax2.annotate(f'Max: {max_het_row["heterogeneity_distance"]:.3f}',
                xy=(max_het_row['param_value'], max_het_row['heterogeneity_distance']),
                xytext=(10, 10), textcoords='offset points',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7),
                fontsize=9)

plt.tight_layout()
plt.savefig('docs/validation/parameter_sweep_segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Optimal Parameter Values

In [ ]:
# Find optimal parameter values (closest to validation targets)
validation_targets = {
    'gini': 0.71,  # Target Gini coefficient
    'top10_share': 35.0,  # Estimated target for top 10 share (%)
    'heterogeneity_distance': 0.35,  # Target heterogeneity (moderate differentiation)
}

optimal_values = []

for param_name, sweep_config in PARAM_SWEEPS.items():
    param_data = df_results[df_results['param_name'] == param_name]
    
    # Calculate composite distance from targets
    distances = []
    for idx, row in param_data.iterrows():
        dist = 0
        for metric, target in validation_targets.items():
            if metric in row:
                dist += ((row[metric] - target) / target) ** 2
        distances.append(np.sqrt(dist))
    
    best_idx = np.argmin(distances)
    best_row = param_data.iloc[best_idx]
    
    optimal_values.append({
        'parameter': param_name,
        'optimal_value': best_row['param_value'],
        'baseline_value': sweep_config['baseline'],
        'deviation': best_row['param_value'] - sweep_config['baseline'],
        'gini': best_row['gini'],
        'heterogeneity': best_row['heterogeneity_distance'],
        'top10_share': best_row['top10_share'],
    })

df_optimal = pd.DataFrame(optimal_values)

# Display optimal values
print("=" * 80)
print("OPTIMAL PARAMETER VALUES (Closest to Validation Targets)")
print("=" * 80)
print("\nValidation Targets:")
for metric, target in validation_targets.items():
    print(f"  {metric}: {target}")

print("\nOptimal Parameter Values:")
print(df_optimal.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(df_optimal))
width = 0.35

bars1 = ax.bar(x - width/2, df_optimal['baseline_value'], width, label='Baseline', color='lightblue')
bars2 = ax.bar(x + width/2, df_optimal['optimal_value'], width, label='Optimal', color='steelblue')

ax.set_ylabel('Parameter Value')
ax.set_title('Baseline vs Optimal Parameter Values\n(Based on Validation Target Proximity)')
ax.set_xticks(x)
ax.set_xticklabels([PARAM_SWEEPS[p]['label'] for p in df_optimal['parameter']], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.2f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('docs/validation/parameter_sweep_optimal_values.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Heterogeneity Deep Dive

In [ ]:
# Detailed heterogeneity analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Heterogeneity vs Gini (trade-off)
ax1 = axes[0, 0]
for param_name in PARAM_SWEEPS.keys():
    param_data = df_results[df_results['param_name'] == param_name]
    ax1.scatter(param_data['gini'], param_data['heterogeneity_distance'], 
               s=100, alpha=0.6, label=param_name)
    # Connect points in order
    ax1.plot(param_data['gini'], param_data['heterogeneity_distance'], 
            alpha=0.3, linewidth=1)

ax1.set_xlabel('Gini Coefficient')
ax1.set_ylabel('Heterogeneity (CV)')
ax1.set_title('Heterogeneity vs Destination Concentration')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.annotate('Ideal region', xy=(0.71, 0.35), xytext=(0.75, 0.45),
            arrowprops=dict(arrowstyle='->', color='green'),
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5),
            fontsize=10)

# 2. Heterogeneity by segment distance distribution
ax2 = axes[0, 1]
baseline_results = df_results[
    (df_results['param_value'] == df_results.groupby('param_name')['param_value'].transform(
        lambda x: x.iloc[np.argmin(np.abs(x - PARAM_SWEEPS[x.name]['baseline']))]
    ))
]

# Get the baseline run (all parameters at baseline)
baseline_run = df_results[df_results['param_name'] == 'softmax_temperature']
baseline_row = baseline_run[baseline_run['param_value'] == 1.0]

if len(baseline_row) > 0:
    baseline_row = baseline_row.iloc[0]
    segment_distances = [baseline_row[f'{seg}_distance'] for seg in segments]
    
    bars = ax2.bar(segments, segment_distances, 
                   color=[segment_colors[seg] for seg in segments])
    ax2.set_ylabel('Average Distance (km)')
    ax2.set_title('Segment Distance Distribution\n(All Parameters at Baseline)')
    ax2.axhline(y=np.mean(segment_distances), color='red', linestyle='--', 
               alpha=0.7, label=f'Mean: {np.mean(segment_distances):.0f} km')
    ax2.legend()
    
    # Add CV annotation
    cv = calculate_heterogeneity_cv({seg: baseline_row[f'{seg}_distance'] for seg in segments})
    ax2.text(0.95, 0.95, f'CV = {cv:.3f}', transform=ax2.transAxes, 
            ha='right', va='top', fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# 3. Heterogeneity vs Parameter Value (all parameters)
ax3 = axes[1, 0]
colors = plt.cm.viridis(np.linspace(0, 1, len(PARAM_SWEEPS)))
for (param_name, sweep_config), color in zip(PARAM_SWEEPS.items(), colors):
    param_data = df_results[df_results['param_name'] == param_name]
    # Normalize parameter value to 0-1 range for comparison
    norm_values = (param_data['param_value'] - param_data['param_value'].min()) / \
                  (param_data['param_value'].max() - param_data['param_value'].min())
    ax3.plot(norm_values, param_data['heterogeneity_distance'], 'o-', 
            linewidth=2, color=color, label=param_name)

ax3.set_xlabel('Normalized Parameter Value (0-1)')
ax3.set_ylabel('Heterogeneity (CV)')
ax3.set_title('Heterogeneity Sensitivity Across All Parameters')
ax3.legend(fontsize=9, ncol=2)
ax3.grid(True, alpha=0.3)

# 4. Interpretation guide
ax4 = axes[1, 1]
ax4.axis('off')

interpretation_text = """
HETEROGENEITY INTERPRETATION GUIDE
===================================

CV > 0.5: HIGH heterogeneity
  - Segments behave very differently
  - Good for resilience/diversity
  - Risk: segments may be unrealistic

CV = 0.2-0.5: MODERATE heterogeneity
  - Balanced segment differentiation
  - Ideal for realistic simulation
  - Target range for validation

CV < 0.2: LOW heterogeneity
  - Segments behave similarly
  - May indicate parameter issue
  - Loss of emergent patterns

KEY FINDINGS:
=============
"""

# Add findings
most_sensitive = df_sensitivity[df_sensitivity['metric'] == 'heterogeneity_distance'].loc[
    df_sensitivity[df_sensitivity['metric'] == 'heterogeneity_distance']['sensitivity_index'].idxmax()
]

findings_text = f"""
• Most sensitive parameter: {most_sensitive['parameter']}
  (Sensitivity index: {most_sensitive['sensitivity_index']:.3f})

• Optimal heterogeneity range: 
  {df_results['heterogeneity_distance'].min():.3f} - {df_results['heterogeneity_distance'].max():.3f}

• Baseline heterogeneity: 
  {df_results[(df_results['param_name']=='softmax_temperature') & 
              (df_results['param_value']==1.0)]['heterogeneity_distance'].values[0]:.3f}
"""

ax4.text(0.1, 0.9, interpretation_text + findings_text, transform=ax4.transAxes,
        fontsize=10, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.savefig('docs/validation/parameter_sweep_heterogeneity_deep_dive.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary and Recommendations

In [ ]:
print("=" * 80)
print("PARAMETER SWEEP ANALYSIS - SUMMARY")
print("=" * 80)

print("\n1. SENSITIVITY RANKINGS (by impact on Gini coefficient):")
gini_ranking = df_sensitivity[df_sensitivity['metric'] == 'gini'].sort_values(
    'sensitivity_index', ascending=False
)
for i, row in gini_ranking.iterrows():
    print(f"   {i+1}. {row['parameter']}: {row['sensitivity_index']:.3f}")

print("\n2. SENSITIVITY RANKINGS (by impact on heterogeneity):")
het_ranking = df_sensitivity[df_sensitivity['metric'] == 'heterogeneity_distance'].sort_values(
    'sensitivity_index', ascending=False
)
for i, row in het_ranking.iterrows():
    print(f"   {i+1}. {row['parameter']}: {row['sensitivity_index']:.3f}")

print("\n3. RECOMMENDED PARAMETER VALUES:")
for _, row in df_optimal.iterrows():
    print(f"   {PARAM_SWEEPS[row['parameter']]['label']}:")
    print(f"      Baseline: {row['baseline_value']}")
    print(f"      Optimal:  {row['optimal_value']} (deviation: {row['deviation']:+.3f})")
    print(f"      Expected Gini: {row['gini']:.3f}, Heterogeneity: {row['heterogeneity']:.3f}")
    print()

print("\n4. KEY INSIGHTS:")
print(f"   • Gini range across all sweeps: {df_results['gini'].min():.3f} - {df_results['gini'].max():.3f}")
print(f"   • Heterogeneity range: {df_results['heterogeneity_distance'].min():.3f} - {df_results['heterogeneity_distance'].max():.3f}")
print(f"   • TFI stress range: {df_results['tfi_stress_pct'].min():.1f}% - {df_results['tfi_stress_pct'].max():.1f}%")

print("\n5. NEXT STEPS:")
print("   • Run fine-grained sweep around optimal values")
print("   • Test interaction effects (full factorial design)")
print("   • Validate against empirical data (UN Tourism, OECD)")
print("   • Document parameter justification in literature_parameters.md")

print(f"\n\nAnalysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nOutput files saved to: docs/validation/")
print(f"  - parameter_sweep_sensitivity.png")
print(f"  - parameter_sweep_global_metrics.png")
print(f"  - parameter_sweep_segment_analysis.png")
print(f"  - parameter_sweep_optimal_values.png")
print(f"  - parameter_sweep_heterogeneity_deep_dive.png")

## 7. Export Results

In [ ]:
# Export full results to CSV
output_file = Path('docs/validation/parameter_sweep_results.csv')
df_results.to_csv(output_file, index=False)
print(f"\nExported full results to: {output_file}")

# Export summary statistics
summary_stats = df_results.groupby('param_name').agg({
    'param_value': ['min', 'max'],
    'gini': ['min', 'max', 'mean', 'std'],
    'heterogeneity_distance': ['min', 'max', 'mean', 'std'],
    'tfi_stress_pct': ['min', 'max', 'mean'],
    'top10_share': ['min', 'max', 'mean'],
}).round(4)

summary_file = Path('docs/validation/parameter_sweep_summary.csv')
summary_stats.to_csv(summary_file)
print(f"Exported summary statistics to: {summary_file}")

# Display summary
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(summary_stats.to_string())